##### This notebook's goal:
To complete and present exploratory data analysis, plotting, and other data mining insights.

##### Author(s):
- (Ahn) Michael Howell - Human Language Technology Masters - Sociolinguist & Agentic AI Engineer - ahn@equita-tech.com
- Cat Xia

---

#### Gather imports

In [51]:
import ast # For ensuring preprocessed data is loaded as intended data types
import numpy as np
import pandas as pd

#### Create helper functions

In [52]:
### AHN

def show_df_details(df, details_to_include = []):
    if "head" in details_to_include:
        print(f"\n** HEAD:\n{df.head()}\n")
    if "tail" in details_to_include:
        print(f"\n** TAIL:\n{df.tail()}\n")
    if "info" in details_to_include:
        print(f"\n** INFO:\n")
        df.info()
        print(f"\n")
    if "describe" in details_to_include:
        print(f"\n** DESCRIBE:\n{df.describe()}\n")
    if "describe-all" in details_to_include:
        print(f"\n** DESCRIBE (ALL):\n{df.describe(include = 'all')}\n")

In [53]:
### CAT

#### Load preprocessed data

In [54]:
## LOAD DATA
preprocessed_df = pd.read_csv(
    "./data/preprocessed_dataset.csv",
    converters={"themes_found": ast.literal_eval})


# To verify our themes_found tuples (str, int) came in as intended:
print(f"\n** Verifying themes_found is tuples:\n")
total_themes = 0
for i, row in preprocessed_df.head(5).iterrows():
    print(f"@ {i}:")
    for theme, count in row["themes_found"]:
        print(f"\t{theme} ({count})")
        total_themes += count
print(f"\nFound {total_themes} themes in the first 5 rows\n")

## MERGE DATAFRAMES
jokes_df = preprocessed_df

## REVIEW DETAILS
show_df_details(jokes_df, ["dtypes", "head", "tail", "info", "describe"])


** Verifying themes_found is tuples:

@ 0:
	ethnicity (1)
@ 1:
	ethnicity (3)
@ 2:
	ethnicity (3)
@ 3:
	gender (1)
	class (1)
@ 4:
	class (2)
	age (1)

Found 12 themes in the first 5 rows


** HEAD:
       id  score  overall_length  distinct_words_count  theme_gender_count  \
0  5tz52q    1.0              22                     7                   0   
1  5tz4dd    0.0              27                    12                   0   
2  5tz319    0.0              40                    18                   0   
3  5tz2wj    1.0             105                    35                   1   
4  5tz1pc    0.0              30                    16                   0   

   theme_gender_bool  theme_ethnicity_count  theme_ethnicity_bool  \
0              False                      1                  True   
1              False                      3                  True   
2              False                      3                  True   
3               True                      0            

#### Additional data prep

In [ ]:
### AHN

## STATIC GLOBALS
THEMES = ['gender','ethnicity','sexuality','ability','age','weight','appearance','class']
COUNT_COL_NAMES = [f"theme_{t}_count" for t in THEMES]

## HELPER FUNCTION
def make_long_df(df, min_count):
    long_df_list = []

    # For each theme, keep rows meeting threshold
    for theme in THEMES:
        included_rows = df[df[f"theme_{theme}_count"] >= min_count].copy()
        included_rows["theme"] = theme
        long_df_list.append(included_rows)

    # Handle rows with no themes (for this threshold)
    mask_any = (df[[f"theme_{t}_count" for t in THEMES]] >= min_count).any(axis = 1)
    none_rows = df[~mask_any].copy()
    none_rows["theme"] = "none"
    long_df_list.append(none_rows)

    # Put all together
    return pd.concat(long_df_list, ignore_index=True)


## GET THEME SETS (BUILD DATAFRAMES)
# Make melted dataframes, which have a "theme" column (rows with multiple themes are duplicated, per theme)
# This is our general dataframe for analysis - it introduces "theme" column for easy coloring, summing, averaging by theme
melted_default = make_long_df(jokes_df, min_count = 1)
# Insightful to have a set which is stricter, requiring at least a couple words to match a theme
melted_strict = make_long_df(jokes_df, min_count = 2)


# Example usage
print(f"\n** HEAD (default):\n{melted_default.head()}\n")
print(f"\n** HEAD (strict):\n{melted_strict.head()}\n")
print(f"\n** SAMPLE (default):\n{melted_default.sample(5, random_state = None)}\n")
print(f"\n** SAMPLE (strict):\n{melted_strict.sample(5, random_state = None)}\n")

# Gather theme sub-dfs easily
print(melted_strict[melted_strict["theme"] == "ethnicity"].head())

# Check everything is as expected
# 1 - melted row counts should not match original joke df; default should have more, strict should be fewer
print(f"\n** ROW COUNTS\nOriginal: {len(jokes_df)} | Melted (default): {len(melted_default)} | Melted (strict): {len(melted_strict)}\n")

# 2 - look at theme distributions briefly
print(f"\n** THEME DISTRIBUTION:\nDefault:\n{melted_default["theme"].value_counts()}\n\nStrict:\n{melted_strict["theme"].value_counts()}\n\n")

# 3 - each melted row should have one theme value (might be "none")
assert "theme" in melted_default.columns
assert melted_default["theme"].notna().all()


** HEAD (default):
       id  score  overall_length  distinct_words_count  theme_gender_count  \
0  5tz2wj    1.0             105                    35                   1   
1  5tz0ef    0.0              13                     9                   1   
2  5tyx6v    3.0             104                    37                  12   
3  5tyt6c    2.0              16                     7                   1   
4  5tyqag   62.0             249                    67                  15   

   theme_gender_bool  theme_ethnicity_count  theme_ethnicity_bool  \
0               True                      0                 False   
1               True                      0                 False   
2               True                      0                 False   
3               True                      0                 False   
4               True                      2                  True   

   theme_sexuality_count  theme_sexuality_bool  ...  theme_age_count  \
0                      0

In [56]:
### CAT

##### Exploratory data analysis (EDA)

In [57]:
### AHN

In [58]:
### CAT

##### Plotting & visual analysis

In [59]:
### AHN
# Each of these plots as default then as strict

# 1 - Theme prevalence - bar plot
# > How common is each theme?

# 2 - Scores by theme - Box plot (log?)
# > Which themes get higher or lower scores?

# 3 - Scores vs number of themes - Box plot
# > Does a joke having more themes correlate with a higher score?

# 4 - Joke length vs. score, colored by theme - Scatter plot
# > Do longer jokes score differently than shorter ones (also showing theme trends)?

In [60]:
### CAT

##### Classification - model training & evaluation

##### Key findings - summary of analysis

#### (Ahn)

##### Exploratory data analysis:
-

##### From plotting & visual analysis:
- 

##### Next directions:
-

##### (Cat)